# 📝 과제 02: 업무 메일을 검토하고 승인받는 AI 비서를 만듭니다

회신 초안을 만들고 사용자의 수정·내용 검토·발송 승인을 받습니다.

<img src="images/assignment02_overview.png" width="1050" alt="업무 메일 AI 비서의 입력, 작업 노드, 분기와 저장 흐름">

**흐름:** 초안 작성 → 내용 검토 → 필요시 수정·재검토 → 발송 승인 → 확정문 저장.

| 단계 | 문항 | 직접 작성할 것 |
|---|---|---|
| State 생성 | 1 | 입력·초안·두 판단·확정문 |
| 노드 생성 | 2~6 | 작성, 검토, 수정, 발송 승인, 확정 |
| 엣지 연결 | 7~8 | 두 라우터와 수정 사이클 |
| 실행·결과 확인 | 9~14 | 스트리밍, 수정, 직접 편집, 승인, 두 반려 경로 |

가상 메일을 사용하며 승인된 메일은 로컬 파일에 저장합니다. 작성·수정 프롬프트는 특정 요일이나 답변을 내장하지 않고 incoming·context·rules를 입력받습니다. 다른 업무 메일에도 같은 그래프를 사용할 수 있습니다.

노드는 입력 State를 직접 변경하지 않고 반환 딕셔너리로 갱신합니다. 자가평가는 노드의 모델 호출과 중단 요청을 고정 응답으로 대체하므로 추가 API 호출 없이 코드 동작을 확인합니다. 메일의 사실관계와 표현은 입력 자료와 직접 대조하세요.

**모델 호출:** 기본 사례의 초안·수정 2회, 내용 반려 사례 1회, 발송 반려 사례 1회로 총 4회입니다. 직접 편집과 승인·반려는 모델을 호출하지 않습니다. 9번부터는 실행 순서를 지켜야 하며 한 사례를 다시 시작하려면 새 ID의 초안 생성부터 진행합니다.

## 준비: 받은 메일과 사용자 상황을 읽습니다

**공통 작성 규칙:** 노드는 입력 State와 그 안의 목록을 직접 변경하지 않고, 갱신할 필드만 반환합니다. 호출 입력은 위치 인자와 키워드 인자 어느 쪽으로 전달해도 됩니다.

**자가평가:** 각 문항을 작성한 뒤 `assert` 셀을 실행하세요. 노드 정의는 제공된 고정 응답으로 입출력을 검사하고, 실행 문항은 실제 실행 결과를 검사합니다. 자가평가 셀 자체는 GPT를 추가 호출하지 않습니다. 통과 후에도 생성된 글의 사실과 표현은 원자료와 직접 대조하세요.

.env와 모델을 준비하는 제공 셀입니다. 그대로 실행하세요. 이 단계에서는 모델을 호출하지 않습니다.

In [ ]:
# 교안과 같은 환경 설정과 모델을 사용합니다. 이 셀은 모델을 호출하지 않습니다.
import os
import json
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict
# 자가평가는 고정 응답으로 노드의 입출력을 검사하며 추가 모델 호출을 하지 않습니다.
from types import SimpleNamespace
from unittest.mock import Mock, patch
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command, interrupt

material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY를 .env에 설정한 뒤 진행하세요.")
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)
print("사용 모델:", llm.model_name)

받은 메일과 사용자 상황을 읽으세요. 가능한 일정과 아직 약속할 수 없는 내용을 확인합니다.

In [ ]:
# 요일·시간·자료 공유 범위는 프롬프트가 아니라 입력 자료에 둡니다.
mail_packet = json.loads((data_dir / "assignment02_mail.json").read_text(encoding="utf-8"))
for name, value in mail_packet.items():
    print(name + ":")
    print(value)
from uuid import uuid4
from langgraph.config import get_stream_writer

## 1. 메일 검토 State를 생성합니다

내용 승인과 발송 승인은 별도 판단입니다.

**작성:** 전체 코드 작성

**제공:** mail_packet과 import를 제공합니다.

- `MailState(TypedDict)` 전체를 작성하세요. incoming, context, rules, draft, decision, feedback, send_decision, approved_mail은 모두 str입니다.
- `mail_input`의 앞 세 필드에는 mail_packet의 동명 값을 넣고 나머지는 빈 문자열로 시작하세요.



<details><summary>힌트</summary>

```text
한 작업의 입력 자료와 검토 과정에서 바뀌는 값을 나누어 생각합니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**결과 확인:** 두 판단과 확정문은 모두 비어 있습니다.

In [ ]:
# [출력 확인]
print("State:", MailState.__annotations__)
print("초기 판단:", repr(mail_input["decision"]), repr(mail_input["send_decision"]))

**1번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 필드 이름·자료형과 초기 입력을 확인합니다.
from typing import get_type_hints
from typing_extensions import is_typeddict
assert is_typeddict(MailState), "MailState를 TypedDict로 선언하세요."
grading_annotations = get_type_hints(MailState)
grading_fields = {"incoming", "context", "rules", "draft", "decision", "feedback", "send_decision", "approved_mail"}
assert set(grading_annotations) == grading_fields, "1번: State의 여덟 필드를 확인하세요."
assert all(grading_type is str for grading_type in grading_annotations.values()), "1번: 모든 필드의 자료형은 str입니다."
assert set(mail_input) == grading_fields, "1번: 초기 입력에도 여덟 필드가 필요합니다."
assert all(mail_input[grading_key] == mail_packet[grading_key] for grading_key in ["incoming", "context", "rules"]), "1번: 입력 자료를 그대로 옮기세요."
assert all(mail_input[grading_key] == "" for grading_key in ["draft", "decision", "feedback", "send_decision", "approved_mail"]), "1번: 입력 자료 이외의 필드는 빈 문자열로 시작합니다."
print("✅ 1번: State와 초기 입력 확인")
print("1번 자가평가 통과")

회신 작성 프롬프트를 준비하세요. 받은 메일·사용자 상황·작성 기준을 입력받습니다.

In [ ]:
# 제공: 업무 상황과 작성 기준을 입력받는 범용 회신 프롬프트입니다.
mail_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 업무 메일 비서입니다. 받은 메일의 요청에 답하는 회신 초안을 작성하세요. "
     "제목과 본문을 쓰고 사용자 상황에 없는 약속은 만들지 마세요.\n작성 기준:\n{rules}"),
    ("human", "받은 메일:\n{incoming}\n사용자 상황:\n{context}"),
])

## 2. 회신 작성 노드를 만듭니다

작성 중인 문장과 진행 상태를 나중에 스트리밍으로 표시합니다.

**작성:** 코드 골격 완성

**제공:** mail_prompt, 진행 알림, 함수 골격을 제공합니다.

- mail_prompt와 llm을 `mail_chain`으로 연결하세요.
- `write_mail`은 incoming·context·rules를 전달하고 응답 text를 draft로 반환하세요.
- 제공된 writer 알림은 유지합니다. 이 노드는 9번 그래프 안에서 실행합니다.

**확인:** 노드 반환은 draft 하나이며 모델 실행은 9번에서 발생합니다.

<details><summary>힌트</summary>

```text
writer가 보내는 진행 정보와 노드의 State 반환값은 역할이 다릅니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# custom 알림은 화면 진행 정보이며 State를 갱신하지 않습니다.
mail_chain = ...
def write_mail(state: MailState):
    """입력 자료로 회신 초안을 작성합니다."""
    writer = get_stream_writer()
    writer({"stage": "writing", "message": "회신 초안을 작성합니다."})
    response = ...
    return ...

**2번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 모델과 진행 알림을 고정 응답으로 대체해 노드만 확인합니다.
# 연결한 체인에 다른 입력을 넣고, 모델에 도착한 프롬프트를 확인합니다.
grading_prompt_input = {"incoming": "공개 문서 요청", "context": "온라인 공유 가능", "rules": "두 문장으로 작성"}
with patch.object(type(llm), "invoke", return_value=SimpleNamespace(text="연결 확인")) as grading_model:
    grading_response = mail_chain.invoke(grading_prompt_input)
    grading_model.assert_called_once()
    assert grading_model.call_args.args[0].to_messages() == mail_prompt.invoke(grading_prompt_input).to_messages(), "2번: 제공 프롬프트의 입력이 모델로 전달되어야 합니다."
    assert grading_response.text == "연결 확인", "2번: 연결한 모델의 응답을 받아야 합니다."
grading_state = {**mail_input, "incoming": "자료 요청", "context": "공개 자료만 공유 가능", "rules": "간결하게 작성"}
grading_before = grading_state.copy()
grading_chain = Mock()
grading_chain.invoke.return_value = SimpleNamespace(text="자가평가용 초안")
grading_writer = Mock()
grading_no_model = Mock()
grading_no_model.invoke.side_effect = AssertionError("2번: mail_chain을 통해 호출하세요.")
with patch.dict(globals(), {"mail_chain": grading_chain, "llm": grading_no_model, "get_stream_writer": Mock(return_value=grading_writer)}):
    grading_result = write_mail(grading_state)
    assert grading_result == {"draft": "자가평가용 초안"}, "2번: 응답 text를 draft 하나로 반환하세요."
    assert grading_chain.invoke.call_count == 1, "체인을 한 번 호출하세요."
    grading_call = grading_chain.invoke.call_args
    grading_payload = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
    assert grading_payload == {"incoming": grading_state["incoming"], "context": grading_state["context"], "rules": grading_state["rules"]}, "지문에 지정한 입력 필드를 체인에 전달하세요."
    grading_writer.assert_called_once_with({"stage": "writing", "message": "회신 초안을 작성합니다."})
    assert grading_state == grading_before, "2번: 입력 State를 직접 바꾸지 마세요."
print("✅ 2번: 작성 노드의 입력·반환·진행 알림 확인")
print("2번 자가평가 통과")

## 3. 내용 검토를 기다리는 노드를 만듭니다

사용자는 수정 요청과 직접 편집 중 적합한 방법을 고릅니다.

**작성:** 코드 골격 완성

**제공:** 검토 요청 딕셔너리와 함수 골격을 제공합니다.

- `review_mail`은 request로 interrupt를 호출하고 응답을 response에 담으세요.
- 응답 action은 approve·edit·reject 중 하나이며 decision에 저장합니다. feedback이 없으면 빈 문자열입니다.
- approve일 때 공백을 제외하고 비어 있지 않은 edited_text가 있으면 원래 문자열을 그대로 draft로 사용합니다. 이외에는 기존 draft를 유지하세요.
- decision·feedback·draft 세 필드만 반환합니다. 응답은 이 실습에서 정한 형식으로 전달합니다.

**확인:** edit은 LLM 수정 요청, approve와 edited_text는 직접 편집 승인입니다. 선택지 표시는 입력값 검증을 대신하지 않습니다.

<details><summary>힌트</summary>

```text
중단 요청을 보여 주는 것과 사람이 보낸 응답을 State로 반영하는 것을 구분합니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 재개하면 검토 노드의 처음부터 실행되므로 중단 전에는 외부 작업을 하지 않습니다.
def review_mail(state: MailState):
    """현재 초안에 대한 사용자의 선택과 편집 내용을 반환합니다."""
    request = {"stage": "content_review", "draft": state["draft"], "actions": ["approve", "edit", "reject"]}
    response = ...
    draft = state["draft"]
    if ...:
        draft = ...
    return ...

**3번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 수정 요청·직접 편집·반려와 빈 편집문을 각각 확인합니다.
grading_cases = [
    ({"action": "approve"}, "기존 초안", ""),
    ({"action": "approve", "edited_text": "  사람이 고친 문장  "}, "  사람이 고친 문장  ", ""),
    ({"action": "approve", "edited_text": ""}, "기존 초안", ""),
    ({"action": "approve", "edited_text": " \n\t "}, "기존 초안", ""),
    ({"action": "edit", "feedback": "더 짧게", "edited_text": "승인 전 편집"}, "기존 초안", "더 짧게"),
    ({"action": "reject", "edited_text": "반려한 편집", "feedback": "내부 재검토"}, "기존 초안", "내부 재검토"),
    ({"action": "approve", "feedback": "내용 확인 완료"}, "기존 초안", "내용 확인 완료"),
]
for grading_response, grading_draft, grading_feedback in grading_cases:
    grading_state = {**mail_input, "draft": "기존 초안"}
    grading_before = grading_state.copy()
    grading_interrupt = Mock(return_value=grading_response)
    grading_no_model = Mock()
    grading_no_model.invoke.side_effect = AssertionError("3번: 검토 노드는 모델을 호출하지 않습니다.")
    with patch.dict(globals(), {"interrupt": grading_interrupt, "llm": grading_no_model, "mail_chain": grading_no_model, "mail_revision_chain": grading_no_model}):
        grading_result = review_mail(grading_state)
        assert grading_result == {"decision": grading_response["action"], "feedback": grading_feedback, "draft": grading_draft}, "3번: action·feedback·edited_text 처리 조건을 확인하세요."
        assert grading_interrupt.call_count == 1, "검토 요청으로 한 번 중단하세요."
        grading_call = grading_interrupt.call_args
        grading_payload = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("value")
        assert grading_payload == {"stage": "content_review", "draft": "기존 초안", "actions": ["approve", "edit", "reject"]}, "제공된 검토 요청을 interrupt에 전달하세요."
        assert grading_state == grading_before, "3번: 입력 State를 직접 바꾸지 마세요."
print("✅ 3번: 검토 응답과 직접 편집의 응답 사례 확인")
print("3번 자가평가 통과")

메일 수정 프롬프트를 준비하세요. 받은 메일·사용자 상황·작성 기준·현재 초안·수정 의견을 입력받습니다.

In [ ]:
# 제공: 수정 의견도 매번 입력으로 전달합니다.
mail_revision_prompt = ChatPromptTemplate.from_messages([
    ("system", "업무 메일 초안을 사용자 의견에 따라 수정하세요. 원래 요청에 대한 답변과 확인된 사실을 유지하세요. "
     "의견에 없는 사실이나 약속을 새로 만들지 마세요.\n작성 기준:\n{rules}"),
    ("human", "받은 메일:\n{incoming}\n사용자 상황:\n{context}\n현재 초안:\n{draft}\n수정 의견:\n{feedback}"),
])

## 4. 수정 노드를 만듭니다

수정한 메일을 자동으로 승인하면 사용자 확인이 생략됩니다.

**작성:** 코드 골격 완성

**제공:** 수정 프롬프트와 진행 알림을 제공합니다.

- `mail_revision_chain`을 연결하세요.
- `revise_mail`은 incoming·context·rules·draft·feedback을 전달하세요.
- 새 text를 draft로, 빈 문자열을 decision으로 반환하세요.

**확인:** 수정 후 초안이 바뀌고 내용 검토 결과는 비워집니다.

<details><summary>힌트</summary>

```text
수정은 새로운 검토 대상이 생기는 단계입니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 수정 후에는 내용 승인을 다시 받아야 합니다.
mail_revision_chain = ...
def revise_mail(state: MailState):
    """의견을 반영한 초안과 비운 내용 검토 결과를 반환합니다."""
    writer = get_stream_writer()
    writer({"stage": "revising", "message": "수정 의견을 반영합니다."})
    response = ...
    return ...

**4번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 원문·상황·기준·초안·의견을 모두 전달하고 내용 판단을 비우는지 확인합니다.
grading_prompt_input = {"incoming": "공개 문서 요청", "context": "온라인 공유 가능", "rules": "간결하게", "draft": "기존 회신", "feedback": "두 문장으로 수정"}
with patch.object(type(llm), "invoke", return_value=SimpleNamespace(text="수정 연결 확인")) as grading_model:
    grading_response = mail_revision_chain.invoke(grading_prompt_input)
    grading_model.assert_called_once()
    assert grading_model.call_args.args[0].to_messages() == mail_revision_prompt.invoke(grading_prompt_input).to_messages(), "4번: 수정 프롬프트의 입력이 모델로 전달되어야 합니다."
    assert grading_response.text == "수정 연결 확인", "4번: 연결한 모델의 응답을 받아야 합니다."
grading_state = {**mail_input, "incoming": "새로운 자료 요청", "context": "다른 담당자의 상황", "rules": "한 문단으로 작성", "draft": "기존 초안", "feedback": "더 간결하게", "decision": "edit"}
grading_before = grading_state.copy()
grading_chain = Mock()
grading_chain.invoke.return_value = SimpleNamespace(text="자가평가용 수정본")
grading_writer = Mock()
grading_no_model = Mock()
grading_no_model.invoke.side_effect = AssertionError("4번: mail_revision_chain을 통해 호출하세요.")
with patch.dict(globals(), {"mail_revision_chain": grading_chain, "mail_chain": grading_no_model, "llm": grading_no_model, "get_stream_writer": Mock(return_value=grading_writer)}):
    grading_result = revise_mail(grading_state)
    assert grading_result == {"draft": "자가평가용 수정본", "decision": ""}, "4번: 수정본을 반환하고 decision을 비우세요."
    assert grading_chain.invoke.call_count == 1, "체인을 한 번 호출하세요."
    grading_call = grading_chain.invoke.call_args
    grading_payload = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
    assert grading_payload == {grading_key: grading_state[grading_key] for grading_key in ["incoming", "context", "rules", "draft", "feedback"]}, "지문에 지정한 입력 필드를 체인에 전달하세요."
    grading_writer.assert_called_once_with({"stage": "revising", "message": "수정 의견을 반영합니다."})
    assert grading_state == grading_before, "4번: 입력 State를 직접 바꾸지 마세요."
print("✅ 4번: 수정 노드의 입력·반환·재검토 준비 확인")
print("4번 자가평가 통과")

## 5. 최종 발송 승인을 기다립니다

내용이 맞아도 발송을 보류할 수 있습니다.

**작성:** 코드 골격 완성

**제공:** 발송 승인 요청을 제공합니다.

- `approve_sending`에서 request로 대기하고, 받은 action을 send_decision으로 반환하세요.
- 이 단계는 approve·reject만 사용하며 첫 번째 decision은 유지합니다.

**확인:** 내용 승인이 끝나도 이 단계에서 한 번 더 멈춥니다.

<details><summary>힌트</summary>

```text
stage는 화면의 검토 단계 이름이며 재개할 대상은 현재 중단 ID로 찾습니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 내용 검토와 발송 승인은 서로 다른 중단 요청입니다.
def approve_sending(state: MailState):
    """현재 메일의 발송 승인 또는 반려를 반환합니다."""
    request = {"stage": "send_approval", "draft": state["draft"], "actions": ["approve", "reject"]}
    response = ...
    return ...

**5번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 내용 승인을 유지하면서 발송 승인과 반려를 따로 반환하는지 확인합니다.
for grading_action in ["approve", "reject"]:
    grading_state = {**mail_input, "draft": "내용 승인된 초안", "decision": "approve"}
    grading_before = grading_state.copy()
    grading_interrupt = Mock(return_value={"action": grading_action})
    grading_no_model = Mock()
    grading_no_model.invoke.side_effect = AssertionError("5번: 발송 승인 노드는 모델을 호출하지 않습니다.")
    with patch.dict(globals(), {"interrupt": grading_interrupt, "llm": grading_no_model, "mail_chain": grading_no_model, "mail_revision_chain": grading_no_model}):
        grading_result = approve_sending(grading_state)
        assert grading_result == {"send_decision": grading_action}, "5번: send_decision만 반환하세요."
        assert grading_interrupt.call_count == 1, "검토 요청으로 한 번 중단하세요."
        grading_call = grading_interrupt.call_args
        grading_payload = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("value")
        assert grading_payload == {"stage": "send_approval", "draft": "내용 승인된 초안", "actions": ["approve", "reject"]}, "제공된 검토 요청을 interrupt에 전달하세요."
        assert grading_state == grading_before, "5번: 기존 내용 판단과 입력 State를 유지하세요."
print("✅ 5번: 별도 발송 승인과 반려 확인")
print("5번 자가평가 통과")

## 6. 승인된 메일을 확정하는 노드를 작성합니다

두 승인이 끝난 결과만 저장 대상이 됩니다.

**작성:** 전체 코드 작성

**제공:** MailState와 함수명 finalize_mail을 제공합니다.

- `finalize_mail(state: MailState)` 전체를 작성하세요.
- draft를 approved_mail 필드 하나로 반환하세요.
- 파일 저장은 12번에서 합니다. 이 노드는 State만 갱신합니다.

**확인:** approved_mail에 승인한 draft가 그대로 담깁니다.

<details><summary>힌트</summary>

```text
두 승인이 모두 필요한 조건은 다음 엣지 연결에서 보장합니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**6번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 승인한 문장을 다시 생성하지 않고 그대로 확정하는지 확인합니다.
grading_state = {**mail_input, "draft": "  승인한 메일\n그대로 보존  ", "decision": "approve", "send_decision": "approve"}
grading_before = grading_state.copy()
grading_no_model = Mock()
grading_no_model.invoke.side_effect = AssertionError("6번: 확정 노드에서 모델을 호출하지 마세요.")
with patch.dict(globals(), {"llm": grading_no_model, "mail_chain": grading_no_model, "mail_revision_chain": grading_no_model}):
    grading_result = finalize_mail(grading_state)
    assert grading_result == {"approved_mail": grading_state["draft"]}, "6번: draft를 그대로 approved_mail 하나로 반환하세요."
    assert grading_state == grading_before, "6번: 입력 State를 직접 바꾸지 마세요."
print("✅ 6번: 승인 문장 보존 확인")
print("6번 자가평가 통과")

## 7. 두 검토의 경로 함수를 작성합니다

같은 approve라도 검토 단계에 따라 다음 목적지가 다릅니다.

**작성:** 전체 코드 작성

**제공:** 함수명과 경로 규칙을 제공합니다.

- `route_mail`은 decision이 approve이면 approve, edit이면 edit, 나머지는 reject를 반환하세요.
- `route_sending`은 send_decision이 approve이면 approve, 나머지는 reject를 반환하세요.
- 두 함수는 State만 읽고 모델을 호출하지 않습니다.



<details><summary>힌트</summary>

```text
자연어 판단은 사용자가 했습니다. 저장된 선택값은 Python으로 분기합니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**결과 확인:** 내용 edit은 수정으로, 발송 reject는 종료로 연결할 수 있습니다.

In [ ]:
# [출력 확인]
for action in ["approve", "edit", "reject"]:
    print("내용:", action, "->", route_mail({**mail_input, "decision": action}))
for action in ["approve", "reject"]:
    print("발송:", action, "->", route_sending({**mail_input, "send_decision": action}))

**7번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 두 라우터가 각각의 판단만 읽는지 확인합니다.
grading_no_model = Mock()
grading_no_model.invoke.side_effect = AssertionError("7번: 라우터는 모델을 호출하지 않습니다.")
with patch.dict(globals(), {"llm": grading_no_model, "mail_chain": grading_no_model, "mail_revision_chain": grading_no_model}):
    for grading_action, grading_expected in [("approve", "approve"), ("edit", "edit"), ("reject", "reject"), ("", "reject"), ("unknown", "reject")]:
        grading_state = {**mail_input, "decision": grading_action, "send_decision": "reject"}
        grading_before = grading_state.copy()
        assert route_mail(grading_state) == grading_expected, "7번: 내용 판단의 세 경로를 확인하세요."
        assert grading_state == grading_before, "7번: 라우터는 State를 읽기만 합니다."
    for grading_action, grading_expected in [("approve", "approve"), ("reject", "reject"), ("edit", "reject"), ("", "reject"), ("unknown", "reject")]:
        grading_state = {**mail_input, "decision": "edit", "send_decision": grading_action}
        grading_before = grading_state.copy()
        assert route_sending(grading_state) == grading_expected, "7번: 발송 판단은 approve 이외에 reject를 반환합니다."
        assert grading_state == grading_before, "7번: 라우터는 State를 읽기만 합니다."
print("✅ 7번: 두 판단의 독립적인 경로 확인")
print("7번 자가평가 통과")

## 8. 노드와 엣지를 연결합니다

검토·수정·최종 승인이 정해진 순서로 실행되어야 합니다.

**작성:** 코드 골격 완성

**제공:** 노드 등록 코드를 제공합니다.

- MailState로 mail_builder를 만드세요. START → write_mail → review_mail로 연결합니다.
- review_mail에서 route_mail을 사용해 approve → approve_sending, edit → revise_mail, reject → END로 연결하세요.
- revise_mail은 review_mail로 돌아갑니다.
- approve_sending에서 route_sending을 사용해 approve → finalize_mail, reject → END로 연결하세요. finalize_mail은 END로 갑니다.
- mail_saver에 InMemorySaver를 만들고 이를 연결해 mail_graph로 컴파일하세요.

**확인:** 두 단계 중 어느 곳에서 반려해도 finalize_mail을 실행하지 않습니다.

<details><summary>힌트</summary>

```text
조건부 분기와 고정 엣지를 같은 출발점에 겹쳐 놓지 않습니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 두 번의 승인과 한 번의 수정 사이클을 연결합니다.
mail_builder = ...
mail_builder.add_node("write_mail", write_mail)
mail_builder.add_node("review_mail", review_mail)
mail_builder.add_node("revise_mail", revise_mail)
mail_builder.add_node("approve_sending", approve_sending)
mail_builder.add_node("finalize_mail", finalize_mail)
...
mail_saver = ...
mail_graph = ...

**8번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 실행 없이 그래프의 연결과 체크포인터를 확인합니다.
grading_graph = mail_graph.get_graph()
assert {"write_mail", "review_mail", "revise_mail", "approve_sending", "finalize_mail"}.issubset(grading_graph.nodes), "8번: 다섯 노드를 등록하세요."
grading_edges = {(grading_edge.source, grading_edge.target, grading_edge.conditional) for grading_edge in grading_graph.edges}
assert grading_edges == {
    (START, "write_mail", False), ("write_mail", "review_mail", False),
    ("review_mail", "approve_sending", True), ("review_mail", "revise_mail", True), ("review_mail", END, True),
    ("revise_mail", "review_mail", False),
    ("approve_sending", "finalize_mail", True), ("approve_sending", END, True),
    ("finalize_mail", END, False),
}, "8번: 두 조건부 분기와 수정 사이클의 연결을 확인하세요."
grading_labels = {(grading_edge.source, grading_edge.data): grading_edge.target for grading_edge in grading_graph.edges if grading_edge.conditional}
assert grading_labels == {("review_mail", "approve"): "approve_sending", ("review_mail", "edit"): "revise_mail", ("review_mail", "reject"): END, ("approve_sending", "approve"): "finalize_mail", ("approve_sending", "reject"): END}, "8번: 각 분기 라벨의 목적지를 확인하세요."
assert isinstance(mail_saver, InMemorySaver) and mail_graph.checkpointer is mail_saver, "8번: mail_saver를 체크포인터로 연결하세요."
print("✅ 8번: 그래프 연결과 메모리 저장 확인")
print("8번 자가평가 통과")

### 스트리밍 출력 영역을 준비합니다

아래 함수는 교안 02의 화면 표시 코드입니다. 작성할 부분은 없습니다. messages는 작성 중인 문장, updates는 노드 변경과 중단, values는 전체 State, custom은 진행 알림에 사용합니다. 한 번의 실행에서 네 종류를 함께 읽고, 받은 종류를 집합으로 반환해 자가평가에 사용합니다.

In [ ]:
# 제공: 모델 문장과 진행 알림이 섞이지 않도록 표시 영역을 분리합니다.
def show_mail_stream(parts):
    """v2 스트림을 읽고 문장, 진행, 현재 상태를 표시합니다."""
    progress_view = display(Markdown("**진행:** 실행 중"), display_id=True)
    draft_view = display(Markdown(""), display_id=True)
    state_view = display({}, display_id=True)
    text_parts = []
    received_modes = set()
    for part in parts:
        received_modes.add(part["type"])
        if part["type"] == "messages":
            message, metadata = part["data"]
            if metadata.get("langgraph_node") in ["write_mail", "revise_mail"] and message.text:
                text_parts.append(message.text)
                draft_view.update(Markdown("".join(text_parts)))
        elif part["type"] == "custom":
            progress_view.update(Markdown("**진행:** " + part["data"]["message"]))
        elif part["type"] == "values":
            state_view.update({"내용 검토": part["data"]["decision"], "발송 승인": part["data"]["send_decision"]})
        elif part["type"] == "updates" and "__interrupt__" in part["data"]:
            progress_view.update(Markdown("**진행:** 사용자 검토 대기"))
    return received_modes

## 9. 초안을 스트리밍하고 현재 검토 요청을 읽습니다

사용자가 전체 작업 종료 전에 글과 진행을 볼 수 있게 합니다.

**작성:** 코드 골격 완성

**제공:** 출력 함수 show_mail_stream과 새 작업 ID를 제공합니다.

- mail_graph의 stream에 mail_input과 mail_config를 전달하세요. 모드는 messages·updates·values·custom, version은 v2입니다.
- 제공 함수를 통해 스트림을 끝까지 읽은 뒤 get_state 결과를 mail_snapshot에 담으세요.
- 현재 interrupts의 첫 요청을 mail_request에 담으세요.



<details><summary>힌트</summary>

```text
stream도 그래프를 실행합니다. 스트리밍 전 invoke를 별도로 호출하지 않습니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 새 메일의 초안을 스트리밍으로 실행하고 첫 검토에서 멈춥니다.
mail_config = {"configurable": {"thread_id": f"mail-{uuid4()}"}}
mail_stream = ...
mail_stream_modes = show_mail_stream(mail_stream)
mail_snapshot = ...
mail_request = ...

**결과 확인:** stage는 content_review, next는 review_mail을 가리킵니다. 두 승인과 approved_mail은 비어 있습니다.

In [ ]:
# [출력 확인]
print("중단 ID:", mail_request.id)
print("검토 단계:", mail_request.value["stage"])
print("다음 노드:", mail_snapshot.next)
display(Markdown(mail_request.value["draft"]))
print("확정문:", repr(mail_snapshot.values["approved_mail"]))

**9번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 스트림을 다시 실행하지 않고 이미 저장된 첫 검토 상태를 읽습니다.
assert mail_stream_modes == {"messages", "updates", "values", "custom"}, "9번: 네 가지 스트림을 모두 받아야 합니다."
grading_snapshot = mail_graph.get_state(mail_config)
assert mail_snapshot.values == grading_snapshot.values, "9번: 스트림을 끝까지 읽은 뒤 상태를 조회하세요."
assert mail_snapshot.next == ("review_mail",), "9번: 내용 검토에서 대기해야 합니다."
assert len(mail_snapshot.interrupts) == 1 and mail_request.id == mail_snapshot.interrupts[0].id, "9번: 현재 첫 중단 요청을 읽으세요."
assert mail_request.value["stage"] == "content_review", "9번: 내용 검토 요청인지 확인하세요."
assert mail_request.value["draft"] == mail_snapshot.values["draft"] and mail_snapshot.values["draft"].strip(), "9번: 생성한 초안이 검토 요청에 있어야 합니다."
assert all(mail_snapshot.values[grading_key] == "" for grading_key in ["decision", "send_decision", "approved_mail"]), "9번: 첫 대기에서는 두 판단과 확정문이 비어 있어야 합니다."
print("✅ 9번: 초안 생성 후 첫 검토 대기 확인. 일정·견적 표현은 직접 읽어 보세요.")
print("9번 자가평가 통과")

## 10. 수정 의견을 전달하고 다시 검토합니다

수정 후 새로 생긴 중단 요청을 읽어야 합니다.

**작성:** 코드 골격 완성

**제공:** 사용자가 보낸 edit_response와 표시 코드를 제공합니다.

- 현재 대기 요청을 mail_request로 다시 읽으세요.
- Command의 resume에는 현재 요청 ID를 키로, edit_response를 값으로 한 딕셔너리를 전달하세요.
- 같은 mail_config와 네 스트림 모드, v2로 재개하세요.
- 수정 후 새로 대기한 요청을 revised_request에 담으세요.



<details><summary>힌트</summary>

```text
thread_id는 유지하고 응답할 중단 ID는 현재 상태에서 새로 읽습니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 현재 중단 ID에 수정 응답을 연결하고 같은 작업을 이어갑니다.
...
edit_response = {"action": "edit", "feedback": "가능한 미팅 시간은 목요일 오전 10시 하나만 제안하고, 견적은 요구사항 확인 후 안내한다고 간결하게 써 주세요."}
revision_stream = mail_graph.stream(
    ..., mail_config,
    stream_mode=["messages", "updates", "values", "custom"], version="v2",
)
revision_stream_modes = show_mail_stream(revision_stream)
...

**결과 확인:** 다시 content_review에서 멈추며, 수정본이 오전 10시 제안과 견적 미확정을 반영해야 합니다.

In [ ]:
# [출력 확인]
print("다시 검토할 단계:", revised_request.value["stage"])
display(Markdown(revised_request.value["draft"]))

**10번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 수정 후 생긴 현재 중단과 State를 확인합니다. 모델은 다시 호출하지 않습니다.
assert revision_stream_modes == {"messages", "updates", "values", "custom"}, "10번: 수정에서도 네 가지 스트림을 모두 받아야 합니다."
grading_snapshot = mail_graph.get_state(mail_config)
assert grading_snapshot.next == ("review_mail",) and len(grading_snapshot.interrupts) == 1, "10번: 수정 후 내용 검토로 돌아가야 합니다."
assert revised_request.id == grading_snapshot.interrupts[0].id and revised_request.id != mail_request.id, "10번: 수정 후의 새 중단 ID를 읽으세요."
assert revised_request.value["stage"] == "content_review", "10번: 수정본을 내용 검토에서 기다려야 합니다."
assert revised_request.value["draft"] == grading_snapshot.values["draft"] and grading_snapshot.values["draft"].strip(), "10번: 현재 수정본을 요청에서 읽으세요."
assert grading_snapshot.values["feedback"] == edit_response["feedback"], "10번: 전달한 수정 의견을 보존해야 합니다."
assert all(grading_snapshot.values[grading_key] == "" for grading_key in ["decision", "send_decision", "approved_mail"]), "10번: 수정 후에는 재검토를 기다리며 확정하지 않습니다."
print("✅ 10번: 수정과 재검토 확인. 오전 10시 제안과 견적 미확정은 직접 읽어 보세요.")
print("10번 자가평가 통과")

## 11. 직접 편집한 메일의 내용을 승인합니다

사용자가 확정한 문장을 다시 생성하지 않아야 합니다.

**작성:** 코드 골격 완성

**제공:** 직접 편집한 edited_text와 승인 응답을 제공합니다. 필요하면 edited_text의 전체 문자열을 직접 고쳐도 됩니다.

- 현재 내용 검토 요청에 content_response로 재개하세요. 결과는 content_result입니다.
- 다음 중단 요청을 send_request에 담으세요.
- draft가 edited_text와 같은지, approved_mail이 여전히 비었는지 확인하세요.



<details><summary>힌트</summary>

```text
첫 승인 뒤에도 발송 승인이 남아 있으므로 그래프는 다시 대기합니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 직접 편집한 문장은 모델에 다시 보내지 않고 그대로 승인합니다.
edited_text = revised_request.value["draft"] + "\n\n추가로 궁금하신 사항이 있으면 회신 부탁드립니다."
content_response = {"action": "approve", "edited_text": edited_text}
...
content_result = ...
...

**결과 확인:** decision은 approve, 새 stage는 send_approval입니다. 모델 추가 호출 없이 직접 편집한 문장이 보존됩니다.

In [ ]:
# [출력 확인]
print("편집문 보존:", content_result["draft"] == edited_text)
print("내용 승인:", content_result["decision"])
print("현재 단계:", send_request.value["stage"])
print("아직 비어 있는 확정문:", repr(content_result["approved_mail"]))

**11번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 직접 편집한 문장이 두 번째 승인 요청까지 그대로 전달됐는지 확인합니다.
grading_snapshot = mail_graph.get_state(mail_config)
assert content_result["draft"] == edited_text and content_result["decision"] == "approve", "11번: 편집문을 그대로 내용 승인하세요."
assert content_result["approved_mail"] == "" and content_result["send_decision"] == "", "11번: 내용 승인만으로 확정하면 안 됩니다."
assert grading_snapshot.next == ("approve_sending",) and len(grading_snapshot.interrupts) == 1, "11번: 발송 승인을 기다려야 합니다."
assert send_request.id == grading_snapshot.interrupts[0].id and send_request.value["stage"] == "send_approval", "11번: 새 발송 승인 요청을 읽으세요."
assert send_request.value["draft"] == edited_text, "11번: 발송 검토 대상도 직접 편집한 문장이어야 합니다."
print("✅ 11번: 편집문 보존과 두 번째 승인 대기 확인")
print("11번 자가평가 통과")

## 12. 발송 승인 후 확정문을 저장합니다

내용 승인과 발송 승인 모두 완료한 문서를 보관합니다.

**작성:** 코드 골격 완성

**제공:** 승인된 메일을 저장하는 출력 코드를 제공합니다.

- 현재 send_approval 요청에 action이 approve인 응답을 전달하세요.
- mail_result에 실행 결과, final_snapshot에 현재 저장 상태를 담으세요.
- approved_mail은 11번 edited_text와 같아야 합니다.



<details><summary>힌트</summary>

```text
마지막 승인에서는 새 초안을 생성하지 않습니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 최종 승인을 받은 결과만 작업 ID별 파일에 저장합니다.
send_request = ...
mail_result = ...
...

**결과 확인:** 두 판단은 approve, next와 interrupts는 비어 있고 승인된 메일 파일이 생성됩니다.

In [ ]:
# [출력 확인]
print("두 판단:", mail_result["decision"], mail_result["send_decision"])
print("편집문과 확정문 일치:", mail_result["approved_mail"] == edited_text)
print("남은 실행:", final_snapshot.next, final_snapshot.interrupts)
display(Markdown(mail_result["approved_mail"]))
mail_path = output_dir / (mail_config["configurable"]["thread_id"] + ".md")
if mail_result["approved_mail"]:
    mail_path.write_text(mail_result["approved_mail"], encoding="utf-8")
    print("저장 파일:", mail_path)

**12번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 이미 승인된 결과·체크포인트·저장 파일을 서로 대조합니다.
assert mail_result["decision"] == "approve" and mail_result["send_decision"] == "approve", "12번: 두 승인이 모두 필요합니다."
assert mail_result["approved_mail"] == edited_text == mail_result["draft"], "12번: 직접 편집한 문장을 그대로 확정하세요."
assert not final_snapshot.next and not final_snapshot.interrupts, "12번: 승인 후에는 남은 실행과 중단이 없어야 합니다."
assert final_snapshot.values["approved_mail"] == edited_text, "12번: 최종 저장 상태를 조회하세요."
assert mail_path.is_file() and mail_path.read_text(encoding="utf-8") == edited_text, "12번: 저장 파일과 승인한 문장이 같아야 합니다."
print("✅ 12번: 최종 승인·종료·파일 저장 확인")
print("12번 자가평가 통과")

### 다른 메일로 두 반려 경로를 확인합니다

같은 그래프에 다른 입력을 사용합니다. 아래는 가상의 자료 공유 요청입니다. 새로운 State 클래스나 프롬프트를 만들 필요가 없습니다.

In [ ]:
# 제공: 구조는 같고 업무 내용만 다른 입력입니다.
other_mail_input = {
    **mail_input,
    "incoming": "협력사에서 내부 제품 로드맵과 공개 제품 소개 자료를 함께 요청했습니다.",
    "context": "공개 제품 소개 자료만 공유할 수 있습니다. 내부 로드맵은 외부 제공 대상이 아닙니다. 자료를 실제 전달하기 전 담당자 승인이 필요합니다.",
}

## 13. 이벤트 스트리밍으로 읽고 내용 단계에서 반려합니다

typed projection 방식으로 같은 HITL 흐름을 읽습니다.

**작성:** 코드 골격 완성

**제공:** v3 문장·상태 표시 루프를 제공합니다. 교안의 고정 패키지 버전을 사용하세요.

- stream_events에 other_mail_input·rejection_config·version v3를 전달해 event_run을 만드세요.
- 제공 루프 실행 뒤 event_run의 output을 event_output, 현재 interrupts의 첫 항목을 event_request로 읽으세요.
- 현재 요청을 reject로 재개하는 stream_events 실행 객체를 rejected_run에 담고 output을 content_rejected로 읽으세요.



<details><summary>힌트</summary>

```text
실행 객체의 output을 읽으면 남은 실행이 진행됩니다. v2의 stream_mode를 지정하지 않습니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# v3는 v2의 type/data 대신 실행 객체에서 결과를 읽습니다.
rejection_config = {"configurable": {"thread_id": f"mail-content-reject-{uuid4()}"}}
event_run = ...
event_text = display(Markdown(""), display_id=True)
event_state = display({}, display_id=True)
for kind, item in event_run.interleave("values", "messages"):
    if kind == "values":
        event_state.update({"내용 검토": item["decision"], "확정문": item["approved_mail"]})
    elif kind == "messages":
        chunks = []
        for text in item.text:
            chunks.append(text)
            event_text.update(Markdown("".join(chunks)))
event_output = ...
event_request = ...
rejected_run = ...
content_rejected = ...

**결과 확인:** 내용 검토에서 반려하면 decision은 reject, send_decision과 approved_mail은 빈 문자열입니다. v3는 실험 단계 API입니다.

In [ ]:
# [출력 확인]
print("첫 대기:", event_run.interrupted, event_request.value["stage"])
display(Markdown(event_output["draft"]))
print("내용 반려:", content_rejected["decision"])
print("발송 판단과 확정문:", repr(content_rejected["send_decision"]), repr(content_rejected["approved_mail"]))
print("남은 실행:", mail_graph.get_state(rejection_config).next)

**13번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# v3 실행 객체의 이미 생성된 결과로 내용 반려를 확인합니다.
assert event_run.interrupted and event_request.id == event_run.interrupts[0].id, "13번: 첫 실행의 중단 요청을 읽으세요."
assert event_request.value["stage"] == "content_review" and event_request.value["draft"] == event_output["draft"], "13번: 다른 메일의 내용 검토 요청을 읽으세요."
assert event_output["incoming"] == other_mail_input["incoming"] and event_output["context"] == other_mail_input["context"], "13번: 다른 업무 입력으로 실행하세요."
assert all(event_output[grading_key] == "" for grading_key in ["decision", "send_decision", "approved_mail"]), "13번: 첫 대기에서는 승인이나 확정이 없어야 합니다."
assert content_rejected == rejected_run.output and content_rejected["decision"] == "reject", "13번: 반려 실행 객체의 결과를 읽으세요."
assert content_rejected["send_decision"] == "" and content_rejected["approved_mail"] == "", "13번: 내용 반려 시 발송 판단과 확정문은 비어 있어야 합니다."
grading_snapshot = mail_graph.get_state(rejection_config)
assert not grading_snapshot.next and not grading_snapshot.interrupts, "13번: 내용 반려 후 종료해야 합니다."
assert not rejected_run.interrupted, "13번: 반려 뒤에는 추가 중단을 기다리지 않습니다."
print("✅ 13번: 이벤트 스트리밍 결과와 내용 반려 확인")
print("13번 자가평가 통과")

## 14. 발송 단계에서 반려해도 확정되지 않는지 확인합니다

내용은 맞지만 지금은 발송하지 않는 선택도 처리해야 합니다.

**작성:** 코드 골격 완성

**제공:** 새 작업 ID와 초안 생성 코드를 제공합니다.

- 현재 내용 검토 요청을 approve로 재개하세요.
- 새로 생긴 발송 승인 요청을 다시 읽고 reject로 재개하세요. 최종 결과는 send_rejected입니다.
- 12번에 승인한 작업의 approved_mail이 그대로인지 별도로 조회하세요.



<details><summary>힌트</summary>

```text
같은 작업의 현재 요청에 답하되 다른 작업의 결과는 바꾸지 않습니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 별도 작업은 내용 승인을 통과한 뒤 발송 단계에서 반려합니다.
send_reject_config = {"configurable": {"thread_id": f"mail-send-reject-{uuid4()}"}}
mail_graph.invoke(other_mail_input, send_reject_config)
request = mail_graph.get_state(send_reject_config).interrupts[0]
...
request = mail_graph.get_state(send_reject_config).interrupts[0]
print("반려할 단계:", request.value["stage"])
send_rejected = ...

**결과 확인:** decision은 approve, send_decision은 reject, approved_mail은 빈 문자열입니다. 12번 작업의 확정문은 유지됩니다.

In [ ]:
# [출력 확인]
print("내용과 발송 판단:", send_rejected["decision"], send_rejected["send_decision"])
print("확정문:", repr(send_rejected["approved_mail"]))
print("남은 실행:", mail_graph.get_state(send_reject_config).next)
print("기존 승인 메일 유지:", mail_graph.get_state(mail_config).values["approved_mail"] == edited_text)

**14번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 발송 반려와 이전 승인 작업의 독립성을 확인합니다.
assert request.value["stage"] == "send_approval", "14번: 발송 승인 단계에서 반려해야 합니다."
assert send_rejected["decision"] == "approve" and send_rejected["send_decision"] == "reject", "14번: 내용 승인과 발송 반려를 구분하세요."
assert send_rejected["approved_mail"] == "", "14번: 발송을 반려하면 확정문은 비어 있어야 합니다."
grading_snapshot = mail_graph.get_state(send_reject_config)
assert not grading_snapshot.next and not grading_snapshot.interrupts, "14번: 반려 후 종료해야 합니다."
assert mail_graph.get_state(mail_config).values["approved_mail"] == edited_text, "14번: 이전 승인 작업의 확정문은 유지해야 합니다."
assert len({mail_config["configurable"]["thread_id"], rejection_config["configurable"]["thread_id"], send_reject_config["configurable"]["thread_id"]}) == 3, "14번: 별도 작업은 서로 다른 ID를 사용하세요."
print("✅ 14번: 발송 반려와 작업별 결과 보존 확인")
print("14번 자가평가 통과")

## 마무리합니다

- 기본 메일의 일정·견적 표현을 입력 자료와 대조하세요.
- 직접 편집한 문장이 최종 결과에 그대로 남는지 확인하세요.
- 내용 반려와 발송 반려에서 확정문이 비었는지 확인하세요.
- 제출: 완성한 노트북, 승인된 메일 파일, 두 반려 결과의 상태 출력.